# Tuning Decision Trees: Preventing Overfitting

**Chapter 5: Advanced Classification Methods**

## Learning Objectives

- Understand overfitting in decision trees
- Learn key hyperparameters for tuning
- Use cross-validation to find optimal parameters
- Visualize the bias-variance tradeoff
- Build a well-tuned decision tree

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsbombpy import sb
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## The Overfitting Problem

**Overfitting** occurs when a model learns the training data too well, including its noise and outliers, and fails to generalize to new data.

**Signs of Overfitting:**
- High training accuracy, low test accuracy
- Very deep, complex tree
- Model memorizes specific examples instead of learning patterns

## Load and Prepare Data

We'll use the same pass completion dataset from Notebook 1.

In [ ]:
# Load data (same as Notebook 1)
matches = sb.matches(competition_id=16, season_id=4)
events = sb.events(match_id=22912)
passes = events[events['type'] == 'Pass'].copy()

# Feature engineering
passes['start_x'] = passes['location'].apply(lambda loc: loc[0] if isinstance(loc, list) else None)
passes['start_y'] = passes['location'].apply(lambda loc: loc[1] if isinstance(loc, list) else None)

def calculate_pass_length(row):
    if pd.isna(row.get('pass_end_location')):
        return None
    start, end = row['location'], row['pass_end_location']
    if isinstance(start, list) and isinstance(end, list):
        return np.sqrt((end[0] - start[0])**2 + (end[1] - start[1])**2)
    return None

passes['pass_length'] = passes.apply(calculate_pass_length, axis=1)
passes['completed'] = passes['pass_outcome'].isna().astype(int)

# Prepare data
features = ['start_x', 'start_y', 'pass_length']
X = passes[features].dropna()
y = passes.loc[X.index, 'completed']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Data prepared: {len(X_train)} training, {len(X_test)} test samples")

## Experiment: Training vs Test Accuracy

Let's train trees with different max_depth values and observe overfitting.

In [ ]:
# Train trees with different depths
train_acc = []
test_acc = []
depths = range(1, 20)

for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    
    train_acc.append(accuracy_score(y_train, tree.predict(X_train)))
    test_acc.append(accuracy_score(y_test, tree.predict(X_test)))

# Plot results
plt.figure(figsize=(12, 6))
plt.plot(depths, train_acc, label='Training Accuracy', marker='o', linewidth=2)
plt.plot(depths, test_acc, label='Test Accuracy', marker='s', linewidth=2)
plt.xlabel('Max Depth', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('The Overfitting Problem: Training vs Test Accuracy', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.axvline(x=depths[np.argmax(test_acc)], color='r', linestyle='--', alpha=0.7, 
            label=f'Best Depth = {depths[np.argmax(test_acc)]}')
plt.legend()
plt.show()

print(f"\nBest max_depth: {depths[np.argmax(test_acc)]} (Test Accuracy: {max(test_acc):.3f})")

## Key Hyperparameters for Tuning

### 1. max_depth
**Controls:** Maximum depth of the tree

**Effect:** Deeper trees can model more complex patterns but risk overfitting

**Typical values:** 3-10

### 2. min_samples_split
**Controls:** Minimum samples required to split a node

**Effect:** Higher values prevent the tree from learning overly specific patterns

**Typical values:** 2-50

### 3. min_samples_leaf
**Controls:** Minimum samples required at a leaf node

**Effect:** Ensures leaves represent meaningful groups, not outliers

**Typical values:** 1-20

### 4. max_features
**Controls:** Number of features to consider for each split

**Effect:** Adds randomness, can improve generalization

**Typical values:** 'sqrt', 'log2', or integer

## Tuning min_samples_split

In [ ]:
# Experiment with min_samples_split
min_samples_values = [2, 5, 10, 20, 50, 100]
cv_scores = []

for min_samples in min_samples_values:
    tree = DecisionTreeClassifier(max_depth=5, min_samples_split=min_samples, random_state=42)
    scores = cross_val_score(tree, X_train, y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())
    print(f"min_samples_split={min_samples:3d}: CV Accuracy = {scores.mean():.3f} (+/- {scores.std():.3f})")

# Plot
plt.figure(figsize=(10, 6))
plt.plot(min_samples_values, cv_scores, marker='o', linewidth=2)
plt.xlabel('min_samples_split', fontsize=12)
plt.ylabel('Cross-Validation Accuracy', fontsize=12)
plt.title('Effect of min_samples_split on Performance', fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()

## Using Cross-Validation for Robust Tuning

Cross-validation splits data into K folds, trains on K-1 folds, and validates on the remaining fold. This process repeats K times, giving a more reliable performance estimate.

In [ ]:
# Grid search over multiple parameters
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [3, 4, 5, 6, 7],
    'min_samples_split': [2, 10, 20],
    'min_samples_leaf': [1, 5, 10]
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.3f}")
print(f"Test accuracy: {grid_search.score(X_test, y_test):.3f}")

## Build the Final Tuned Tree

In [ ]:
# Train with best parameters
best_tree = grid_search.best_estimator_

# Evaluate
y_pred = best_tree.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Final Model Performance:")
print(f"Training Accuracy: {best_tree.score(X_train, y_train):.3f}")
print(f"Test Accuracy: {accuracy:.3f}")
print(f"\nModel Complexity:")
print(f"Number of leaves: {best_tree.tree_.n_leaves}")
print(f"Actual depth: {best_tree.tree_.max_depth}")

## Summary

In this notebook, we:

1. ✅ Understood overfitting (high training accuracy, low test accuracy)
2. ✅ Learned key hyperparameters (max_depth, min_samples_split, min_samples_leaf)
3. ✅ Used cross-validation for robust evaluation
4. ✅ Applied grid search to find optimal parameters
5. ✅ Built a well-tuned decision tree

## Key Takeaways

- **Overfitting is the main challenge** with decision trees
- **max_depth is the most important** hyperparameter
- **Cross-validation** provides reliable performance estimates
- **Grid search** automates hyperparameter tuning
- **Balance complexity and performance** - simpler is often better

## Next Steps

Single trees are still unstable. In the next notebook, we'll learn how **Random Forests** combine many trees for robust predictions!

## Practice Exercises

1. **Tune max_features**: Add max_features to the grid search
2. **Different Metrics**: Use 'roc_auc' instead of 'accuracy' for scoring
3. **Visualize Best Tree**: Plot the tree with optimal parameters
4. **Feature Engineering**: Add more features and see if performance improves
5. **Randomized Search**: Try RandomizedSearchCV for faster tuning